In [48]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr, col, lit
spark = SparkSession.builder.appName("Jupyter").getOrCreate()

events = spark.read.option("header", "true") \
                .csv("/home/iceberg/data/events.csv") \
                .withColumn("event_date", expr("DATE_TRUNC('day', event_time)"))
devices = spark.read.option("header","true").csv("/home/iceberg/data/devices.csv")

df = events.join(devices,on="device_id",how="left")
df = df.withColumnsRenamed({'browser_type': 'browser_family', 'os_type': 'os_family'})

In [51]:
sorted = df.repartition(10, col("event_date"))\
    .sortWithinPartitions(col("event_date"), col("host"))\
    .withColumn("event_time", col("event_time").cast("timestamp")) 

sortedTwo = df.repartition(10, col("event_date"))\
    .sort(col("event_date"), col("host"))\
    .withColumn("event_time", col("event_time").cast("timestamp")) 

sorted.select('event_date, event_time, device_id, user_id, referrer, host, url'.split(', ')).show()
sortedTwo.select('event_date, event_time, device_id, user_id, referrer, host, url'.split(', ')).show()


+-------------------+--------------------+-----------+-----------+--------------------+--------------------+--------------------+
|         event_date|          event_time|  device_id|    user_id|            referrer|                host|                 url|
+-------------------+--------------------+-----------+-----------+--------------------+--------------------+--------------------+
|2021-01-07 00:00:00|2021-01-07 09:21:...|  532630305| 1129583063|                NULL|admin.zachwilson....|                   /|
|2021-01-07 00:00:00|2021-01-07 02:58:...| 1088283544| -648945006|                NULL|    www.eczachly.com|                   /|
|2021-01-07 00:00:00|2021-01-07 04:17:...| -158310583|-1871780024|                NULL|    www.eczachly.com|                   /|
|2021-01-07 00:00:00|2021-01-07 10:03:...| 1088283544|  203689086|                NULL|    www.eczachly.com|/blog/what-exactl...|
|2021-01-07 00:00:00|2021-01-07 18:45:...|  532630305|-1180485268|                NULL|   

In [ ]:
# .sortWithinPartitions() sorts within partitions, whereas .sort() is a global sort, which is very slow

# Note - exchange is synonymous with Shuffle

In [17]:
sorted = df.repartition(10, col("event_date"))\
    .sortWithinPartitions(col("event_date"), col("host"))\
    .withColumn("event_time", col("event_time").cast("timestamp")) 

sortedTwo = df.repartition(10, col("event_date"))\
    .sort(col("event_date"), col("host"))\
    .withColumn("event_time", col("event_time").cast("timestamp")) 

sorted.explain()
sortedTwo.explain() 

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [device_id#332, user_id#331, referrer#333, host#334, url#335, cast(event_time#336 as timestamp) AS event_time#1207, event_date#343, browser_family#388, os_family#389, device_type#372]
   +- Sort [event_date#343 ASC NULLS FIRST, host#334 ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(event_date#343, 10), REPARTITION_BY_NUM, [plan_id=2382]
         +- Project [device_id#332, user_id#331, referrer#333, host#334, url#335, event_time#336, event_date#343, browser_type#370 AS browser_family#388, os_type#371 AS os_family#389, device_type#372]
            +- BroadcastHashJoin [device_id#332], [device_id#369], LeftOuter, BuildRight, false
               :- Project [user_id#331, device_id#332, referrer#333, host#334, url#335, event_time#336, date_trunc(day, cast(event_time#336 as timestamp), Some(Etc/UTC)) AS event_date#343]
               :  +- FileScan csv [user_id#331,device_id#332,referrer#333,host#334,url#335,e

In [32]:
%%sql

CREATE DATABASE IF NOT EXISTS bootcamp

++
||
++
++

In [56]:
%%sql

DROP TABLE IF EXISTS bootcamp.events

++
||
++
++

In [57]:
%%sql

DROP TABLE IF EXISTS bootcamp.events_sorted

++
||
++
++

In [58]:
%%sql

DROP TABLE IF EXISTS bootcamp.events_unsorted

++
||
++
++

In [59]:
%%sql

CREATE TABLE IF NOT EXISTS bootcamp.events (
    url STRING,
    referrer STRING,
    browser_family STRING,
    os_family STRING,
    device_family STRING,
    host STRING,
    event_time TIMESTAMP,
    event_date DATE
)
USING iceberg
PARTITIONED BY (years(event_date));


++
||
++
++

In [60]:
%%sql


CREATE TABLE IF NOT EXISTS bootcamp.events_sorted (
    url STRING,
    referrer STRING,
    browser_family STRING,
    os_family STRING,
    device_family STRING,
    host STRING,
    event_time TIMESTAMP,
    event_date DATE
)
USING iceberg
PARTITIONED BY (years(event_date));

++
||
++
++

In [61]:
%%sql


CREATE TABLE IF NOT EXISTS bootcamp.events_unsorted (
    url STRING,
    referrer STRING,
    browser_family STRING,
    os_family STRING,
    device_family STRING,
    host STRING,
    event_time TIMESTAMP,
    event_date DATE
)
USING iceberg
PARTITIONED BY (year(event_date));

++
||
++
++

In [128]:
start_df = df.repartition(4, col("event_date")).withColumn("event_time", col("event_time").cast("timestamp")) \
    
first_sort_df = start_df.sortWithinPartitions(col("event_date"), col('browser_family'), col("host"))

start_df.write.mode("overwrite").saveAsTable("bootcamp.events_unsorted")
first_sort_df.write.mode("overwrite").saveAsTable("bootcamp.events_sorted")

In [127]:
%%sql

SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files, 'sorted' 
FROM demo.bootcamp.events_sorted.files

UNION ALL
SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files, 'unsorted' 
FROM demo.bootcamp.events_unsorted.files


size,num_files,sorted
5091464,4,sorted
5553022,4,unsorted


In [31]:
%%sql
SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files FROM demo.bootcamp.events.files;

size,num_files
None,0


In [ ]:
%%sql 
SELECT COUNT(1) FROM bootcamp.matches_bucketed.files

count(1)
3665
